In [ ]:
%matplotlib module://ipympl.backend_nbagg
import SimpleITK as sitk
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


import sys
sys.path.insert(0, "/workspace/segmentation")
from tools import img_show3D, window, img_show2D, img_show3D_interactive


In [ ]:
img_dir = Path ("/workspace/segmentation/data/practice/image_quadra/QUADRA_HC_021/retest_CT-AC.nii.gz")
img = sitk.ReadImage(img_dir)

mask_dir = Path ("/workspace/segmentation/data/practice/mask_quadra/QUADRA_HC_021/retest/liver.nii.gz")
mask = sitk.ReadImage(mask_dir)

In [ ]:


def harris_detector (img, smooting_rad_mm = 1, integration_rad_mm = 1, k = 0.005):

    #initial smooting - removing noise
    smooting_filter = sitk.DiscreteGaussianImageFilter()
    smooting_filter.SetVariance(smooting_rad_mm ** 2)
    smooting_filter.SetUseImageSpacing(True)

    smooth_img = smooting_filter.Execute(img)

    #computing the gradient
    gradient_filter = sitk.GradientImageFilter()
    gradient_filter.SetUseImageSpacing(True)
    gradient_filter.SetUseImageDirection(False)

    gradient_img = gradient_filter.Execute(smooth_img)

    gradient_x = sitk.VectorIndexSelectionCast(gradient_img, 0)
    gradient_y = sitk.VectorIndexSelectionCast(gradient_img, 1)
    gradient_z = sitk.VectorIndexSelectionCast(gradient_img, 2)

    #create gaussian filter to accunt neighboring voxel when computing the vector M
    gaussian_harris = sitk.DiscreteGaussianImageFilter()
    gaussian_harris.SetVariance(integration_rad_mm ** 2)
    gaussian_harris.SetUseImageSpacing(True)

    # cumputing vertor M
    #harris smooting - considering surounding voxel when calculating harris response
    # A, D, F: locally averaged squared gradients along x, y, and z.
    # They describe the strength of variation along each axis.
    # B, C, E: locally averaged products between different gradient components.
    # They describe how those components occur together.
    A = gaussian_harris.Execute(gradient_x * gradient_x)
    B = gaussian_harris.Execute(gradient_x * gradient_y)
    C = gaussian_harris.Execute(gradient_x * gradient_z)
    D = gaussian_harris.Execute(gradient_y * gradient_y)
    E = gaussian_harris.Execute(gradient_y * gradient_z)
    F = gaussian_harris.Execute(gradient_z * gradient_z)

    #compute one scalar value per voxel.
    
    det_M = A*D*F + 2.0*B*C*E - A*E*E - D*C*C - F*B*B
    # penalise the gradient magnitude
    trace_M = A + D + F

    # k harris constant, the bigger the higer we penalise the gradient magnitude
    harris_response = det_M - k * trace_M * trace_M * trace_M

    return harris_response

In [ ]:
def local_maxima (harris_response, radius_mm = 1,  ):
    
    harris_resp_vec = sitk.GetArrayFromImage(harris_response)
    
    #calculating the search radius in voxel
    spacings = np.asarray(harris_response.GetSpacing())
    radius_voxels =  np.maximum(np.ceil(radius_mm/spacings), 1) #set the minimum radius as 1 voxel

    #calculating the search kernel size
    kernel_size = tuple(int(2 * radius + 1) for radius in radius_voxels)

    #computing the local maxima
    #create a copy image array, but all values in side the kernel are replaced by the highest value in that kernel
    maxima = maximum_filter(harris_resp_vec, size= kernel_size, mode= "nearest")

    #Extracting the coordinate of the local maxima
    #criteria:
    #the value of a voxel in maxima must the same with harris respon, indicate the initial location of the highest value
    # the voxel value must above threshold, so only high gradient are selected
    # the coordinate is inside the mask

    threshold = float(np.quantile(positive, float(quantile)))
